# P5 — Auditoria encadeada e explicabilidade

Demonstração dos módulos da Pessoa 5:

- `sentinel.audit.chain` — log append-only com encadeamento SHA-256 (`HashChainAuditSink`).
- `sentinel.narrative` — fatos estruturados do alerta, texto determinístico, validação do texto
  gerado por LLM e fallback.

**O que é mock aqui:** o previsor e o detector são os mocks do backend (`mock-persistence`,
`mock-threshold`), os dados são sintéticos (`sentinel.synthetic`) e o alerta "completo" (com região,
CUSUM e contribuições) é inventado (`sentinel.narrative.mock`). Nada disso é detecção de vazamento.
Na seção 4 o "LLM" é uma função falsa; a seção 5 chama o Granite no watsonx.ai de verdade, só se
as variáveis `WATSONX_*` estiverem definidas.

In [1]:
import json
import os
import shutil
import sys
import tempfile
from datetime import UTC, datetime
from pathlib import Path

# Na VM: REPO = Path.home() / "shared" / "civicwater" / "users" / "<seu-nome>" / "repo"
# Local: sobe a partir da pasta do notebook até achar o pyproject.toml do repositório.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(REPO / "src"))  # importa o SEU clone

for v in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"]:
    os.environ[v] = "1"

%load_ext autoreload
%autoreload 2

import sentinel

print("sentinel de:", Path(sentinel.__file__).parent)

sentinel de: C:\Users\leomi\OneDrive\Área de Trabalho\IBM-TechXchange-2026-hackaton\Team-CodigoMaster_IBM-Z-Datathon-2026---Civic-Water-Sentinal\src\sentinel


In [2]:
from sentinel.audit import HashChainAuditSink, audit_path_for, verify_alert, verify_file
from sentinel.config import Settings
from sentinel.main import build_pipeline
from sentinel.narrative import FunctionNarrator, WatsonxNarrator, build_prompt, explain
from sentinel.narrative.mock import mock_alert
from sentinel.schemas.alerts import AuditEvent
from sentinel.schemas.windows import SensorWindow
from sentinel.sensors import SensorRegistry
from sentinel.synthetic import generate

# Arquivos da demo numa pasta temporária (fora do Git).
WORK = Path(tempfile.mkdtemp(prefix="p5-audit-"))
print("arquivos em:", WORK)


def show(obj):
    print(json.dumps(obj, indent=2, ensure_ascii=False, default=str))

arquivos em: C:\Users\leomi\AppData\Local\Temp\p5-audit-34io4h0q


## 1. A cadeia sozinha

Cada evento vira uma linha JSON com `seq`, `payload_sha256`, `prev_hash` e `hash`:

`hash = SHA-256(prev_hash ‖ JSON canônico do registro sem prev_hash e hash)`

In [3]:
sink = HashChainAuditSink(WORK / "demo.jsonl")
for i in range(3):
    ref = sink.append(
        AuditEvent(
            type="window.scored",
            window_end=datetime(2019, 1, 1, 0, 5 * i, tzinfo=UTC),
            payload={"status": "scored", "i": i},
        )
    )
    print(ref)

show(sink.records()[1])
print(sink.verify())
print("head (guarde fora da VM):", sink.head)

seq=0 hash='e5a1a9c0c148fb2fc11bd7ec0eda3fb01f7b57f7ac9cbcca2b14864597401d26'
seq=1 hash='e20cb77a1133be18e543fe4760fe00fe290a5b0d3ef488321dff5e8199f43f73'
seq=2 hash='03333bf9314c1aed418118ed210f75f2198ea52af5d14de5b0f780cce1e421be'
{
  "alert_id": null,
  "hash": "e20cb77a1133be18e543fe4760fe00fe290a5b0d3ef488321dff5e8199f43f73",
  "payload": {
    "i": 1,
    "status": "scored"
  },
  "payload_sha256": "993ee9ccd1a9cd3e975de012695c822e24002407405c8663cb2844145020d79f",
  "prev_hash": "e5a1a9c0c148fb2fc11bd7ec0eda3fb01f7b57f7ac9cbcca2b14864597401d26",
  "record_version": "1.0",
  "recorded_at": "2026-10-17T19:28:44.050024Z",
  "seq": 1,
  "type": "window.scored",
  "window_end": "2019-01-01T00:05:00Z"
}
ChainVerification(ok=True, records=3, head=AuditRef(seq=2, hash='03333bf9314c1aed418118ed210f75f2198ea52af5d14de5b0f780cce1e421be'), broken_at=None, reason=None)
head (guarde fora da VM): seq=2 hash='03333bf9314c1aed418118ed210f75f2198ea52af5d14de5b0f780cce1e421be'


## 2. Adulteração

Alterar, remover ou reordenar um registro quebra a cadeia a partir dele. Reescrever o arquivo
inteiro de forma consistente só é detectado comparando com o *head* guardado fora (âncora).

In [4]:
anchor = sink.head
lines = (WORK / "demo.jsonl").read_text().splitlines()

alterado = WORK / "alterado.jsonl"
rec = json.loads(lines[1])
rec["payload"]["status"] = "insufficient_history"
alterado.write_text("\n".join([lines[0], json.dumps(rec), lines[2]]) + "\n")
print("payload alterado:", verify_file(alterado))

removido = WORK / "removido.jsonl"
removido.write_text("\n".join([lines[0], lines[2]]) + "\n")
print("registro removido:", verify_file(removido))

forjado = HashChainAuditSink(WORK / "forjado.jsonl")  # atacante recalcula tudo
for i in range(3):
    forjado.append(
        AuditEvent(
            type="window.scored",
            window_end=datetime(2019, 1, 1, 0, 5 * i, tzinfo=UTC),
            payload={"status": "forjado", "i": i},
        )
    )
print("reescrita consistente, sem âncora:", verify_file(WORK / "forjado.jsonl").ok)
print("reescrita consistente, com âncora:", verify_file(WORK / "forjado.jsonl", anchor))

payload alterado: ChainVerification(ok=False, records=1, head=AuditRef(seq=0, hash='e5a1a9c0c148fb2fc11bd7ec0eda3fb01f7b57f7ac9cbcca2b14864597401d26'), broken_at=1, reason='payload_sha256 does not match the payload (payload altered)')
registro removido: ChainVerification(ok=False, records=1, head=AuditRef(seq=0, hash='e5a1a9c0c148fb2fc11bd7ec0eda3fb01f7b57f7ac9cbcca2b14864597401d26'), broken_at=1, reason='sequence break: expected seq 1, found 2')
reescrita consistente, sem âncora: True
reescrita consistente, com âncora: ChainVerification(ok=False, records=2, head=AuditRef(seq=1, hash='9c008a0dd8059f81e01e7d5678e3447595fbc4da38fe3dd1e110968ff1208f41'), broken_at=2, reason='hash differs from the external anchor for seq 2')


## 3. Ligada ao pipeline do backend (mocks)

Um dia sintético (288 janelas) com um degrau de pressão em 3 sensores a partir da janela 200.
O backend chama `sink.append(...)` para cada janela e cada alerta; aqui só trocamos o sink.

Num replay de verdade na VM, o arquivo fica no local combinado (fora dos clones do Git):
`audit_path_for("<run>")` → `~/shared/civicwater/audit/<run>.jsonl`. A demo usa a pasta temporária.

In [5]:
print("local combinado na VM:", audit_path_for("replay-2019"))

settings = Settings()  # 288 / 144 / 300 s
registry = SensorRegistry.load(REPO / "config" / "sensors.provisional.json")
chain = HashChainAuditSink(audit_path_for("demo-pipeline", root=WORK))
pipeline = build_pipeline(settings, registry=registry, audit_sink=chain)

for payload in generate(
    datetime(2019, 3, 14, tzinfo=UTC), 288, registry,
    seed=7, noise_m=0.05, leak_at=200, leak_sensors=registry.pressure[:3],
):
    pipeline.ingest(SensorWindow.model_validate(payload))

local combinado na VM: C:\Users\leomi\shared\civicwater\audit\replay-2019.jsonl


In [6]:
# Pega o primeiro alerta pelo registro alert.raised da própria cadeia.
raised = [r for r in chain.records() if r["type"] == "alert.raised"]
print("eventos na cadeia:", len(chain.records()), "| alertas:", len(raised))
alert = pipeline.alerts.get(raised[0]["alert_id"])
print("alerta:", alert.alert_id, "| audit:", alert.audit)
print("cadeia:", chain.verify())
print("alerta confere com o registro:", verify_alert(alert, chain.get(alert.audit.seq)) is None)

eventos na cadeia: 289 | alertas: 1
alerta: A-20190314T1640Z-1 | audit: seq=200 hash='015b8fb8b2137281f11667c71020f1eefddcee8cd9a9a7eb0039720e8d6e8a63'
cadeia: ChainVerification(ok=True, records=289, head=AuditRef(seq=288, hash='557b3a47402698726080775c550df285811bd4a5e219fa340c3f950e9fcec620'), broken_at=None, reason=None)
alerta confere com o registro: True


## 4. Explicação

Sem LLM: texto determinístico (sempre disponível). Com LLM: o texto só é aceito se todo sensor,
região, número, data e horário estiver no payload do alerta; senão volta o template.
Idioma padrão: **inglês** (`language="pt"` continua disponível).

In [7]:
e = explain(alert)  # alerta do pipeline com mocks: sem região nem CUSUM
print(e.text)
print("\nfonte:", e.source, "| payload_sha256 == registro:",
      e.alert_payload_sha256 == chain.get(alert.audit.seq)["payload_sha256"])

[SIMULATION] Produced by mock components: not a real leak detection.
Alert A-20190314T1640Z-1: suspected leak (region not estimated).
Evaluated window: 2019-03-14 16:40 UTC.
Estimated deviation onset: 2019-03-14 16:40 UTC, 0.00 h before the window.
Deviating sensors (observed - expected): P03 -5.06 m (z -5.06); P02 -4.96 m (z -4.96); P01 -4.95 m (z -4.95).
Models: forecaster mock-persistence@0.1.0; detector mock-threshold@0.1.0.
Audit record seq 200.
This explanation restates the detection engine's decision; it does not decide whether a leak exists.

fonte: template | payload_sha256 == registro: True


In [8]:
rico = mock_alert()  # inventado: região, CUSUM, início e contribuições
print(explain(rico).text)
print("\n--- prompt enviado ao LLM ---\n")
print(build_prompt(rico))

[SIMULATION] Produced by mock components: not a real leak detection.
Alert A-20190314T1230Z-1: suspected leak in region R4.
Evaluated window: 2019-03-14 12:30 UTC.
Estimated deviation onset: 2019-03-14 02:10 UTC, 10.33 h before the window.
Deviating sensors (observed - expected): P14 -0.91 m (z -4.10); P21 -0.62 m (z -3.20); P09 -0.48 m (z -2.70).
CUSUM 12.40 against threshold 8.00.
Main forecaster contributions: P14: hour_of_week +0.21 m, F01_inflow -0.12 m, P14_lag_1 +0.08 m; P21: hour_of_week +0.17 m, F02_inflow -0.09 m.
Models: forecaster example-forecaster@0.0.0; detector example-cusum@0.0.0.
Audit record seq 150.
This explanation restates the detection engine's decision; it does not decide whether a leak exists.

--- prompt enviado ao LLM ---

You write short explanations for operators of a water distribution network.
The detection engine has already made the decision: the alert below is a SUSPECTED leak.
You do not decide, confirm or rule out leaks.

Rules:
1. Use only the senso

In [9]:
# "LLMs" falsos: um texto correto e um que inventa sensor, região, número e horário.
bom = FunctionNarrator("llm-falso@0", lambda prompt: (
    "[SIMULATION] Alert A-20190314T1230Z-1 reports a suspected leak in region R4. "
    "Sensors P14 and P21 read 0.91 m and 0.62 m below the expected pressure since "
    "2019-03-14 02:10 UTC. The CUSUM statistic reached 12.4 against a threshold of 8.0."
))
alucinado = FunctionNarrator("llm-falso@0", lambda prompt: (
    "[SIMULATION] Confirmed leak in region R2: sensor P07 dropped 7.3 m at 03:15."
))

for narrador in (bom, alucinado):
    e = explain(rico, narrador)
    print("fonte:", e.source, "| fallback:", e.fallback_reason)
    print("check:", e.check)
    print(e.text, "\n")

narrative rejected


fonte: llm | fallback: None
check: ok=True unknown_references=() unknown_numbers=() unknown_datetimes=() forbidden_phrases=() missing_mock_label=False
[SIMULATION] Alert A-20190314T1230Z-1 reports a suspected leak in region R4. Sensors P14 and P21 read 0.91 m and 0.62 m below the expected pressure since 2019-03-14 02:10 UTC. The CUSUM statistic reached 12.4 against a threshold of 8.0. 

fonte: template | fallback: narrative_rejected
check: ok=False unknown_references=('R2', 'P07') unknown_numbers=('7.3',) unknown_datetimes=('03:15',) forbidden_phrases=('Confirmed leak',) missing_mock_label=False
[SIMULATION] Produced by mock components: not a real leak detection.
Alert A-20190314T1230Z-1: suspected leak in region R4.
Evaluated window: 2019-03-14 12:30 UTC.
Estimated deviation onset: 2019-03-14 02:10 UTC, 10.33 h before the window.
Deviating sensors (observed - expected): P14 -0.91 m (z -4.10); P21 -0.62 m (z -3.20); P09 -0.48 m (z -2.70).
CUSUM 12.40 against threshold 8.00.
Main foreca

## 5. watsonx.ai / Granite de verdade

`WatsonxNarrator` chama a API de chat do watsonx.ai só com a biblioteca padrão (nada a instalar
no LinuxONE). Ele lê a configuração do ambiente: `WATSONX_APIKEY`, `WATSONX_URL` (ex.:
`https://us-south.ml.cloud.ibm.com`), `WATSONX_PROJECT_ID` e `WATSONX_MODEL_ID`. Em 2026-10-17 o
único Granite de chat é `ibm/granite-4-h-small`, só em us-south e eu-de (ver
`docs/audit-and-narrative.md`).

**Nunca escreva a API key no notebook nem a commite.** Para defini-la só nesta sessão do kernel:

```python
import getpass
os.environ["WATSONX_APIKEY"] = getpass.getpass("WATSONX_APIKEY: ")
```

Sem as variáveis, a célula abaixo só avisa e segue. Se o Granite falhar ou inventar algo, `explain`
devolve o template e registra o motivo em `fallback_reason`.

In [10]:
if all(os.environ.get(k) for k in
       ("WATSONX_APIKEY", "WATSONX_URL", "WATSONX_PROJECT_ID", "WATSONX_MODEL_ID")):
    granite = WatsonxNarrator.from_env()
    for alvo in (rico, alert):
        e = explain(alvo, granite)
        print("fonte:", e.source, "| modelo:", e.generator, "| fallback:", e.fallback_reason)
        print("check:", e.check)
        print(e.text, "\n")
else:
    print("Variáveis WATSONX_* não definidas: chamada real ao Granite pulada.")

Variáveis WATSONX_* não definidas: chamada real ao Granite pulada.


## 6. Âncora externa e verificação por linha de comando

Ao fim de um replay, anote o *head* fora da VM e verifique depois:

```bash
python -m sentinel.audit.verify <arquivo.jsonl> --anchor-seq <seq> --anchor-hash <hash>
```

In [11]:
print("head:", chain.head)
print(f"python -m sentinel.audit.verify {chain.path} --anchor-seq {chain.head.seq} --anchor-hash {chain.head.hash}")

head: seq=288 hash='557b3a47402698726080775c550df285811bd4a5e219fa340c3f950e9fcec620'
python -m sentinel.audit.verify C:\Users\leomi\AppData\Local\Temp\p5-audit-34io4h0q\demo-pipeline.jsonl --anchor-seq 288 --anchor-hash 557b3a47402698726080775c550df285811bd4a5e219fa340c3f950e9fcec620


In [ ]:
shutil.rmtree(WORK)  # limpa os arquivos da demo